In [18]:
import pandas as pd
import numpy as np

df=pd.read_csv("data/RELIANCE.csv",parse_dates=["Date"],index_col="Date").sort_index()
df.head()

,Symbol,Series,Prev Close,Open,High,Low,Last,Close,VWAP,Volume,Turnover,Trades,Deliverable Volume,%Deliverble
Date,,,,,,,,,,,,,,
2000-01-03,RELIANCE,EQ,233.05,237.50,251.70,237.50,251.70,251.70,249.37,4456424,1.111319e+14,NaN,NaN,NaN
2000-01-04,RELIANCE,EQ,251.70,258.40,271.85,251.30,271.85,271.85,263.52,9487878,2.500222e+14,NaN,NaN,NaN
2000-01-05,RELIANCE,EQ,271.85,256.65,287.90,256.65,286.75,282.50,274.79,26833684,7.373697e+14,NaN,NaN,NaN
2000-01-06,RELIANCE,EQ,282.50,289.00,300.70,289.00,293.50,294.35,295.45,15682286,4.633254e+14,NaN,NaN,NaN
2000-01-07,RELIANCE,EQ,294.35,295.00,317.90,293.00,314.50,314.55,308.91,19870977,6.138388e+14,NaN,NaN,NaN


In [19]:
#

df["Daily_Return_%"]=df["Close"].pct_change()*100
df["SMA_10_Return"]=df["Daily_Return_%"].rolling(window=10).mean()
df["Volatility_20d"]=df["Daily_Return_%"].rolling(window=20).std()

# to create a shifted binary target (1 = tomorrow clawa higher , 0= lower/flat)
df["Target_Direction"]=np.where(df["Daily_Return_%"].shift(-1)>0,1,0)

#dropping nan
df.dropna(inplace=True)

features=["Daily_Return_%","SMA_10_Return","Volatility_20d"]
x=df[features]
y=df["Target_Direction"]
print("cleaned rows \n ",len(x))

cleaned rows 
  2456


In [20]:
from sklearn.model_selection import TimeSeriesSplit
# 5 chorocological folds
tscv=TimeSeriesSplit(n_splits=5)

print("chronological fold boundaries")
for fold,(train_idx,test_idx) in enumerate(tscv.split(x),1):
    train_start,train_end=x.index[train_idx[0]].date(),x.index[train_idx[-1]].date()
    test_start,test_end=x.index[test_idx[0]].date(),x.index[test_idx[-1]].date()

    print("Fold",fold ,"| Train:" ,train_start,"to",train_end,"--->  Test:",test_start,"to",test_end) 

chronological fold boundaries
Fold 1 | Train: 2011-06-01 to 2013-01-21 --->  Test: 2013-01-22 to 2014-09-12
Fold 2 | Train: 2011-06-01 to 2014-09-12 --->  Test: 2014-09-15 to 2016-05-17
Fold 3 | Train: 2011-06-01 to 2016-05-17 --->  Test: 2016-05-18 to 2018-01-05
Fold 4 | Train: 2011-06-01 to 2018-01-05 --->  Test: 2018-01-08 to 2019-09-05
Fold 5 | Train: 2011-06-01 to 2019-09-05 --->  Test: 2019-09-06 to 2021-04-30


In [21]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

quant_pipeline=Pipeline(
    steps=[
        ("scaler",StandardScaler()),
        ("model",RandomForestClassifier(n_estimators=100,max_depth=5,random_state=42))

    ]
)

# run chronological cross validation
#pass cv=tscv instead of integer like cv=5
ts_scores=cross_val_score(quant_pipeline,x,y,cv=tscv,scoring="accuracy")

print("Individual fold accuracues:",np.round(ts_score*100,2),"%")

mean_acc=round(np.mean(ts_scores)*100,2)
std_acc=round(np.std(ts_scores)*100,2)

print("Mean Accuracy:",mean_acc,"%(±",std_acc,"%)")

Individual fold accuracues: [47.43 48.17 44.74 54.03 48.9 ] %
Mean Accuracy: 48.66 %(± 3.03 %)
